# Lyapunov heatmap — trajectories starting on the satellite

Same map as `lyapunov_heatmap.ipynb` (sign of the largest Lyapunov exponent over $\varepsilon_p \times \varepsilon_d/g_2$, same parameters,
`t_max = 30/κ_b`, transient `Ttr = 15/κ_b`). The difference is the initial state: a point of the **satellite** limit cycle, so that a
trajectory that is not chaotic ends on the satellite. Adapted from `analysis/chaos/chiral_cat/comparison_heatmap_satellites.ipynb`.

**12/10/2026:** $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (static pump), instead of $J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$. At a given $\varepsilon_d/g_2$ the drive is about twice as large as before. The map (cells up to the JSON export) was recomputed; the check cells below it were not, and their saved outputs and `_12_10_2026` files still use the $J_1$ definition.

**Initial state (22/10/2026, report v2 "TO DO for v3").** The trajectory starts exactly on the satellite: the $\tau = 0$ point of the exact period-1
orbit from Hill's method (harmonic balance, $N_H = 16$), computed on the grid of the map by `Hill_satellite_seed_grid.ipynb`
(`satellite_seed_hill_22_10_2026.json`). With the HB state $[x_a, x_b, dx_a/d\tau, dx_b/d\tau]$ in $x^\lambda$ units,
$$a(0) = \frac{x_a + i\,dx_a/d\tau}{\sqrt2\,\varphi_a}, \qquad b(0) = \frac{x_b + i\,(dx_b/d\tau)/2}{\sqrt2\,\varphi_b}.$$
**Lower threshold in $g_2$: the fold.** The satellite is born, together with a saddle orbit, at a fold (saddle-node of limit cycles), from
84.4 MHz at $\varepsilon_d/g_2 = 2$ down to 32.5 MHz at $\varepsilon_d/g_2 = 13.5$ (40.4 MHz at 10.12). Below the fold there is no satellite and the
points are left blank. The seed grid was obtained by pseudo-arclength down to the fold in each column, so it never continues onto the saddle;
the fold line is drawn dashed on the maps.

Before 22/10/2026 the seed was the analytic satellite (appendix "Satellites", `satellite_hb`, files `_14_10_2026`). Its existence boundary is 8–15 %
below the true fold, and between the fold and about $1.2\times$ the fold it is too far from the orbit and ends on the cat, so the lower edge of the
satellite region measured the quality of the seed, not where the satellite exists. With the exact seed, a stable satellite is never left:
below the period doubling ($\approx 75$ MHz at 10.12) the map shows where the satellite is an attractor, above it what the satellite turns into.
`satellite_hb` is kept below for the check cells, which still use the analytic seed.

**End state.** Starting near the satellite does not guarantee ending on it, e.g. at $\varepsilon_p = 0.45$, $\varepsilon_d = 10.12\,g_2$ the trajectory
falls onto the cat. Each trajectory is therefore also integrated to $t_\text{max} + T_\text{tr}$ and classified from $\langle|b|\rangle$ over the
last memory period ($\approx 0.1$ on the cat, $\approx 1.6$–$2.1$ on the satellite): satellite, cat, or chaotic ($\lambda > 0$).

In [ ]:
using MPSDynamics, ChaosTools, ProgressBars

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0

In [ ]:
t_max = 30/κ_b
t_w = 15/κ_b*20                 # transient discarded by lyapunov (Ttr)
T_a = 2π/ω_a                 # memory period

environment = MeanField(κ_a, κ_b)
system_0 = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, 0.0, 0.0, 0.0)
environment_prepared = prepare(system_0, environment)
dt = get_dt(system_0, environment)

# Analytic satellite (appendix "Satellites", full system) at (ϵ_p, ϵ_d/g2); member s = ±1 of the pair a → -a.
# Returns the harmonic amplitudes in the paper's units (a_paper = φ_a a, b_paper = φ_b b), or nothing if no root with x² > 0.
function satellite_hb(ϵ_p, ϵ_d_g2_ratio; s = 1, ȳ_grid = range(1e-3, 3.0, length = 3000))
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b
    β_a′ = 2E_J*sin(ϵ_p)*φ_a^2/ω_a
    β_b′ = 2E_J*sin(ϵ_p)*φ_b^2/ω_b
    ϵ_d′ = ϵ_d_g2_ratio*g2*φ_b/ω_b
    κ_b′ = κ_b/ω_b
    β_Σ = β_a′ + β_b′
    γ = 1 + β_b′/(β_a′*(1 + κ_b′^2))

    ϵ_eff(y) = sqrt(max(ϵ_d′^2 - 4κ_b′^2*y^2, 0.0))
    function y_of(ȳ)                       # y = ȳ + β_a′ ϵ_eff(y)/(2β_b′), fixed point (κ_b′ ≪ 1)
        y = ȳ + β_a′*ϵ_d′/(2β_b′)
        for _ in 1:30
            y = ȳ + β_a′*ϵ_eff(y)/(2β_b′)
        end
        return y
    end
    u_of(ȳ) = (η = 2β_b′*ȳ; (sqrt((1 + η)^2 + 4η^2) - (1 + η))/(2η))
    x²_of(ȳ) = (u = u_of(ȳ); (2(1 + u)*ȳ^2 - ϵ_eff(y_of(ȳ))/β_b′)/(1 + 2u))
    resid(ȳ) = (u = u_of(ȳ); ȳ*(1 + u) - 2β_Σ*((1 + (1 + u)^2/2)*ȳ^2 + (1 + u^2)*x²_of(ȳ)))

    for k in 1:length(ȳ_grid)-1                # first physical root on the grid, refined by bisection
        lo, hi = ȳ_grid[k], ȳ_grid[k+1]
        resid(lo)*resid(hi) > 0 && continue
        for _ in 1:60
            mid = (lo + hi)/2
            resid(lo)*resid(mid) <= 0 ? (hi = mid) : (lo = mid)
        end
        ȳ = (lo + hi)/2
        x² = x²_of(ȳ)
        x² > 0 || continue
        u, y, x = u_of(ȳ), y_of(ȳ), sqrt(x²)
        2κ_b′*abs(y) <= ϵ_d′ || continue     # phase locking requires 2κ_b′|y| ≤ ϵ_d′
        θ = -asin(2κ_b′*y/ϵ_d′)/2
        a_1 = -(1 + u)*ȳ/(2γ)
        return (a_0 = s*x*cis(θ), a_1 = a_1, a_m1 = -β_a′*ϵ_eff(y)/(2β_b′)*cis(2θ),
                b_0 = y*cis(2θ), b_m1 = s*u*x*cis(3θ), b_2 = β_b′/β_a′*a_1)
    end
    return nothing
end

# laboratory-frame amplitudes at t = 0 (τ = 0), in the simulation's units
satellite_initial_amplitudes(S) = ((S.a_0 + S.a_1 + S.a_m1)/φ_a, (S.b_0 + S.b_m1 + S.b_2)/φ_b)

# SystemATS starting at a(0), b(0)
system_from_amplitudes(a_0, b_0) = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb,
    sqrt(2*Z_a)*real(a_0), sqrt(2/Z_a)*imag(a_0), sqrt(2*Z_b)*real(b_0), sqrt(2/Z_b)*imag(b_0))

In [ ]:
ϵ_p_list = vcat(0.2:.01:1.1)
ϵ_d_g2_ratio_list = 2:.1:13.5   # was 5:.1:13.5; extended to overlap the OTOC grid

# Exact satellite (Hill) at τ = 0 on this grid, from Hill_satellite_seed_grid.ipynb. seed_state[j][i] = [x_a, x_b, dx_a/dτ, dx_b/dτ] (x^λ units)
# for ϵ_d/g2 index j and ϵ_p index i; nothing below the satellite fold.
using JSON
seed_data = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/satellite_seed_hill_22_10_2026.json")
@assert Float64.(seed_data["eps_p_list"]) ≈ ϵ_p_list && Float64.(seed_data["eps_d_over_g2"]) ≈ collect(ϵ_d_g2_ratio_list)
seed_state = seed_data["state"]
g2_sat_fold_MHz = Float64.(seed_data["g2_fold_MHz"])   # fold of the satellite (its birth), per ϵ_d/g2

# a(0), b(0) of the exact satellite, or nothing below the fold
function hill_satellite_amplitudes(j, i)
    st = seed_state[j][i]
    isnothing(st) && return nothing
    x_a, x_b, v_a, v_b = Float64.(st)
    return (x_a + 1im*v_a)/(sqrt(2)*φ_a), (x_b + 1im*v_b/2)/(sqrt(2)*φ_b)
end

In [ ]:
using Statistics
Lyap_coef_chaos = fill(NaN, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))   # NaN: below the satellite fold
mean_abs_b_end = fill(NaN, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))    # <|b|> over the last memory period
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_end = t_max + t_w
t_save_end = range(t_end - T_a, t_end, length = 60)
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    drive.ϵ_0 = ϵ_p_i
    g2_i = sin(ϵ_p_i)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
    for j in 1:length(ϵ_d_g2_ratio_list)
        seed = hill_satellite_amplitudes(j, i)
        isnothing(seed) && continue            # below the fold: no satellite
        a_0, b_0 = seed
        system = system_from_amplitudes(a_0, b_0)
        system_prepared = prepare(system)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

        drive.ϵ_d = ϵ_d_g2_ratio_list[j]*g2_i
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        Lyap_coef_chaos[j,i] = lyapunov(CoupledODEs(equation_1ODE!, vcat(u_0, u_dot_0), p), t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)

        # end state: same trajectory integrated to t_max + Ttr, last memory period only
        ϕ_a, ϕ_b, q_a, q_b, _ = simulator(t_end, system, drive, environment; t_save = t_save_end)
        mean_abs_b_end[j,i] = mean(@. sqrt(ϕ_b^2/(2Z_b) + q_b^2*Z_b/2))
    end
end

In [ ]:
Lyap_coef_chaos_sign = map(x -> isnan(x) ? NaN : (x > 0 ? 1.0 : -1.0), Lyap_coef_chaos)

b_threshold = 0.8   # <|b|> on the last period: ≈ 0.1 on the cat, ≈ 1.6–2.1 on the satellite
# 1 = satellite, 2 = cat, 3 = chaotic, NaN = below the satellite fold
end_state = map(Lyap_coef_chaos, mean_abs_b_end) do λ, b
    isnan(λ) ? NaN : (λ > 0 ? 3.0 : (b > b_threshold ? 1.0 : 2.0))
end
n_seed = count(!isnan, end_state)
println("points above the satellite fold (Hill seed): ", n_seed, "/", length(end_state))
for (k, name) in ((1, "satellite"), (2, "cat"), (3, "chaotic"))
    println(rpad(name, 10), round(count(==(k), end_state)/n_seed, digits=3))
end

In [ ]:
g2_MHz_list = E_J.*sin.(ϵ_p_list).*φ_a^2 .*φ_b/2π*1e3
α_sq = ϵ_d_g2_ratio_list./2

heatmap(g2_MHz_list, α_sq, mean_abs_b_end, cmap=:viridis,
        xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\varepsilon_d/2g_2",
        colorbar_title=L"\langle|b|\rangle", title=L"\langle|b|\rangle"*" over the last memory period",
        size=(1.2*size_px_car[1], size_px_car[2]), bottom_margin=5mm, left_margin=5mm)


In [ ]:
using JSON
α_sq = (ϵ_d_g2_ratio_list./2)
g2_list = E_J.*sin.(ϵ_p_list).*φ_a^2 .*φ_b

# instability of the period-2 orbit from Hill_method_doubling_corrected.ipynb (along the ascending ϵ_d/g2 stored next to it)
hill_path = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_12_10_2026_Hill_unstable.json"
hill_ok = isfile(hill_path)
hill_ok || println("No Hill line: run the 2D sweep and the save cell of Hill_method_doubling_corrected.ipynb to create ", hill_path)

function add_axes!(p)
    hill_ok && plot!(p, Float64.(JSON.parsefile(hill_path)), Float64.(JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/eps_d_ratio_list_12_10_2026_Hill_unstable.json"))./2, color=:black, lw=2, label="")
    plot!(p, g2_sat_fold_MHz, α_sq, color=:black, ls=:dash, lw=1.5, label="")   # birth of the satellite (fold)
    plot!(p, xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\varepsilon_d/2g_2",
        xlims=(minimum(g2_list/2π*1e3),maximum(g2_list/2π*1e3)), ylims=(minimum(α_sq),maximum(α_sq)))
end

p_lyap = heatmap(g2_list/2π*1e3, α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds, clims=(-1, 1), colorbar=false,
                 title="Sign of Lyapunov exp.")
add_axes!(p_lyap)

end_colors = [RGB(13/256,76/256,116/256), RGB(134/256,187/256,216/256), RGB(185/256,21/256,21/256)]   # satellite, cat, chaotic
p_end = heatmap(g2_list/2π*1e3, α_sq, end_state, color=cgrad(end_colors, 3, categorical=true), clims=(0.5, 3.5),
                colorbar=false, title="End state")
add_axes!(p_end)
for (k, name) in enumerate(("satellite", "cat", "chaotic"))   # legend for the categories
    scatter!(p_end, [NaN], [NaN], color=end_colors[k], markershape=:square, msw=0, label=name)
end
plot!(p_end, [NaN], [NaN], color=:black, ls=:dash, lw=1.5, label="satellite fold")
plot!(p_end, legend=:bottomleft)

p_map = plot(p_lyap, p_end, layout=(1, 2), size=(2*size_px_car[1], size_px_car[2]), bottom_margin=5mm, left_margin=5mm)
savefig(p_map, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/heatmap_lyapunov_satellite_start_22_10_2026.pdf")
p_map

Same grid and file format as `data_lyapunov_07_04_2026.json` (outer index: ϵ_p). NaN entries are written as `null`.

In [ ]:
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/data_lyapunov_satellite_start_22_10_2026.json", "w") do io
    JSON.print(io, Lyap_coef_chaos_sign)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/end_state_satellite_start_22_10_2026.json", "w") do io
    JSON.print(io, end_state)       # 1 satellite, 2 cat, 3 chaotic, NaN below the satellite fold
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_satellite_start_22_10_2026.json", "w") do io
    JSON.print(io, g2_list)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/alpha_sq_list_satellite_start_22_10_2026.json", "w") do io
    JSON.print(io, α_sq)
end

## Checks: magnitude of λ and parameter continuation

The cells below only need the setup cells above (up to the grid definition), not the full map. They still start from the **analytic** seed
(`satellite_hb`), not from the exact one, and were not re-run after 12/10/2026.

**Helpers.** `run_from_state` integrates from a state $(\phi_a, q_a, \phi_b, q_b)$ to $t_\text{end} = nT_a$. The drive (period $T_a/2$) and the static pump
are then back in phase, so the end state can seed the next run, again started at $t = 0$. The attractor is classified from the stroboscopic
samples $a(nT_a)$ over the last $40\,T_a$ and from $\langle|b|\rangle$ over the last period:
cat ($\langle|b|\rangle < 0.8$), satellite (period $T_a$), period $2T_a$, chaotic ($\lambda > 0$), or other with $\lambda \le 0$ (quasi-periodic, higher period, or not yet settled).
Both periods are detected with a relative tolerance of $10^{-2}$ on the samples. Near a bifurcation, critical slowing down can leave an orbit unsettled after
$30/\kappa_b$, so it shows up as "other".

In [ ]:
using Statistics, JSON
g2_of(ϵ_p) = sin(ϵ_p)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
g2_MHz(ϵ_p) = g2_of(ϵ_p)/2π*1e3

drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)              # holds a reference to drive: set_drive! updates both
function set_drive!(ϵ_p, ϵ_d_g2_ratio)
    drive.ϵ_0 = ϵ_p
    drive.ϵ_d = ϵ_d_g2_ratio*g2_of(ϵ_p)
end

# state (ϕ_a, q_a, ϕ_b, q_b) of SystemATS, from the complex amplitudes and back
state_from_amplitudes(a, b) = (sqrt(2Z_a)*real(a), sqrt(2/Z_a)*imag(a), sqrt(2Z_b)*real(b), sqrt(2/Z_b)*imag(b))
system_from_state(s) = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, s...)

# Integrates from state s at t = 0 to t_end = n T_a ≥ T. The drive (period T_a/2) and the static pump are back in phase at t_end,
# so the end state can seed the next run, started again at t = 0.
# Returns the end state, a at the last n_strobe multiples of T_a (stroboscopic samples) and <|b|> over the last memory period.
n_strobe = 40
n_per_period = 20
function run_from_state(s, ϵ_p, ϵ_d_g2_ratio; T = 30/κ_b)
    set_drive!(ϵ_p, ϵ_d_g2_ratio)
    t_end = ceil(T/T_a)*T_a
    t_save = range(t_end - (n_strobe - 1)*T_a, t_end, length = (n_strobe - 1)*n_per_period + 1)
    ϕ_a, ϕ_b, q_a, q_b, _ = simulator(t_end, system_from_state(s), drive, environment; t_save = t_save)
    a_strobe = @. ϕ_a[1:n_per_period:end]/sqrt(2Z_a) + 1im*q_a[1:n_per_period:end]*sqrt(Z_a/2)
    mean_abs_b = mean(@. sqrt(ϕ_b[end-n_per_period:end]^2/(2Z_b) + q_b[end-n_per_period:end]^2*Z_b/2))
    return (ϕ_a[end], q_a[end], ϕ_b[end], q_b[end]), a_strobe, mean_abs_b
end

# largest Lyapunov exponent from state s (same settings as the map)
function lyap_from_state(s, ϵ_p, ϵ_d_g2_ratio; T = t_max, Ttr = t_w)
    set_drive!(ϵ_p, ϵ_d_g2_ratio)
    system_prepared = prepare(system_from_state(s))
    u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    return lyapunov(CoupledODEs(equation_1ODE!, vcat(u_0, u_dot_0), p), T, Δt = dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = Ttr)
end

# 1 satellite (period T_a), 2 cat, 3 period 2T_a, 4 other non-chaotic (quasi-periodic, higher period, not yet settled), 5 chaotic.
# λ is only needed for the attractors that are neither period 1 nor period 2.
b_threshold = 0.8   # <|b|> on the last period: ≈ 0.1 on the cat, ≈ 1.6–2.1 on the satellite
P_tol = 1e-2        # tolerance on the stroboscopic samples, relative to <|a|>
function classify(a_strobe, mean_abs_b, λ = NaN)
    mean_abs_b < b_threshold && return 2.0
    tol = P_tol*mean(abs, a_strobe)
    maximum(abs, a_strobe .- a_strobe[end]) < tol && return 1.0
    maximum(abs, a_strobe[1:end-2] .- a_strobe[3:end]) < tol && return 3.0
    return λ > 0 ? 5.0 : 4.0
end
class_colors = [RGB(13/256,76/256,116/256), RGB(134/256,187/256,216/256), RGB(46/256,139/256,87/256),
                RGB(230/256,160/256,40/256), RGB(185/256,21/256,21/256)]   # satellite, cat, period 2, other non-chaotic, chaotic
class_names = ("satellite", "cat", "period 2", "other, λ ≤ 0", "chaotic")


# Hill instability of the period-2 orbit, interpolated at a given ϵ_d/g2 (same file as the map)
hill_path = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_12_10_2026_Hill_unstable.json"
hill_ok = isfile(hill_path)
hill_g2 = hill_ok ? Float64.(JSON.parsefile(hill_path)) : Float64[]
hill_ratio = range(5, 13.5, 40)
function hill_at(r)
    k = clamp(searchsortedlast(hill_ratio, r), 1, length(hill_ratio) - 1)
    w = (r - hill_ratio[k])/step(hill_ratio)
    return (1 - w)*hill_g2[k] + w*hill_g2[k+1]
end

# smallest ϵ_p of a list for which the analytic satellite exists
hb_edge(ϵ_p_path, ϵ_d_g2_ratio) = something(findfirst(ϵ_p -> !isnothing(satellite_hb(ϵ_p, ϵ_d_g2_ratio)), ϵ_p_path), length(ϵ_p_path) + 1)

### Magnitude of λ along $\varepsilon_d = 10.12\,g_2$

The map only shows the sign of $\lambda$. Just past a Neimark–Sacker bifurcation the motion is quasi-periodic, with $\lambda = 0$, so the sign right at
the Hill line can be numerical noise. Here $\lambda/\kappa_b$ is computed with the same protocol as the map (start on the analytic satellite,
$T_\text{tr} = 15/\kappa_b$) for $t_\text{max} = 30, 100, 300/\kappa_b$:

- chaos on an attractor: $\lambda > 0$ and stable as $t_\text{max}$ grows;
- quasi-periodic: $\lambda \to 0$ as $t_\text{max}$ grows;
- transient chaos: $\lambda$ decreases with $t_\text{max}$ and turns negative once the trajectory escapes to the cat or the satellite.

In [ ]:
ratio_cut = 10.12
ϵ_p_cut = 0.56:0.005:0.84
t_max_list = [30, 100, 300]./κ_b
λ_cut = fill(NaN, length(t_max_list), length(ϵ_p_cut))   # NaN: no satellite solution
for i in ProgressBar(1:length(ϵ_p_cut))
    S = satellite_hb(ϵ_p_cut[i], ratio_cut)
    isnothing(S) && continue
    s_0 = state_from_amplitudes(satellite_initial_amplitudes(S)...)
    for k in 1:length(t_max_list)
        λ_cut[k, i] = lyap_from_state(s_0, ϵ_p_cut[i], ratio_cut; T = t_max_list[k], Ttr = t_w)
    end
end

g2_cut = g2_of.(ϵ_p_cut)/2π*1e3
p_λ_cut = plot(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\lambda/\kappa_b",
               title=L"\varepsilon_d = "*string(ratio_cut)*L"\,g_2"*", start on the satellite")
for k in 1:length(t_max_list)
    plot!(p_λ_cut, g2_cut, λ_cut[k, :]/κ_b, marker=:circle, ms=2, msw=0,
          label=L"t_{max} = "*string(round(Int, t_max_list[k]*κ_b))*L"/\kappa_b")
end
hline!(p_λ_cut, [0], color=:gray, ls=:dash, label="")
hill_ok && vline!(p_λ_cut, [hill_at(ratio_cut)], color=:black, lw=2, label="Hill instability")
plot!(p_λ_cut, size=(1.5*size_px_car[1], size_px_car[2]), legend=:bottomright, bottom_margin=5mm, left_margin=5mm)
savefig(p_λ_cut, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/lyapunov_magnitude_cut_satellite_start_12_10_2026.pdf")
p_λ_cut

### Parameter continuation along $\varepsilon_d = 10.12\,g_2$

The satellite settled at $\varepsilon_p = 0.55$ ($g_2/2\pi \approx 36$ MHz) is followed up and down in $\varepsilon_p$ (step 0.005). Each point starts
from the end state of the previous one and runs for $30/\kappa_b$. $\lambda$ is computed at every point from that end state, with $T_\text{tr} = 0$ and $t_\text{max} = 30/\kappa_b$.

- **Going down**, the point where the satellite falls on the cat is the true existence edge (saddle-node). It does not depend on the analytic
  amplitudes. If it lies left of the analytic edge, the satellite exists in the light-blue band of the end-state map and the analytic seed was just outside its basin.
- **Going up**, the stroboscopic samples give a bifurcation diagram: one point for the satellite, two after period doubling, a closed curve on a torus, a cloud in chaos.
  It shows whether the followed attractor loses regularity at the Hill line, and whether it later falls on the cat (boundary crisis).

In [ ]:
# Follows the attractor reached from state s_0 along ϵ_p_path at fixed ϵ_d/g2: each point starts from the end state of the previous one.
# stop_on_cat: stop once the attractor is the cat (going down in ϵ_p the cat is the only attractor left, the remaining points stay NaN).
# with_lyap: compute λ at every point (otherwise only where the stroboscopic samples are neither period 1 nor period 2).
function continuation(s_0, ϵ_p_path, ϵ_d_g2_ratio; stop_on_cat = false, with_lyap = false, T_lyap = t_max)
    n = length(ϵ_p_path)
    state_class, λ = fill(NaN, n), fill(NaN, n)
    a_strobe = fill(complex(NaN), n, n_strobe)
    s = s_0
    for (i, ϵ_p) in enumerate(ϵ_p_path)
        s, a_strobe[i, :], mean_abs_b = run_from_state(s, ϵ_p, ϵ_d_g2_ratio)
        if with_lyap || classify(a_strobe[i, :], mean_abs_b) == 4.0
            λ[i] = lyap_from_state(s, ϵ_p, ϵ_d_g2_ratio; T = T_lyap, Ttr = 0.0)   # s is already on the attractor
        end
        state_class[i] = classify(a_strobe[i, :], mean_abs_b, λ[i])
        stop_on_cat && state_class[i] == 2.0 && break
    end
    return state_class, λ, a_strobe
end

# Settled satellite at (ϵ_p, ϵ_d/g2) from the analytic seed, or nothing if the seed does not end on it
function satellite_state(ϵ_p, ϵ_d_g2_ratio; T = 60/κ_b)
    S = satellite_hb(ϵ_p, ϵ_d_g2_ratio)
    isnothing(S) && return nothing
    s, a_strobe, mean_abs_b = run_from_state(state_from_amplitudes(satellite_initial_amplitudes(S)...), ϵ_p, ϵ_d_g2_ratio; T = T)
    return classify(a_strobe, mean_abs_b) == 1.0 ? s : nothing
end

In [ ]:
ϵ_p_start = 0.55                                    # g2/2π ≈ 36 MHz, inside the satellite band at ϵ_d = 10.12 g2
ϵ_p_up = ϵ_p_start:0.005:0.84
ϵ_p_down = ϵ_p_start:-0.005:0.30
s_start = satellite_state(ϵ_p_start, ratio_cut)
isnothing(s_start) && error("the analytic seed does not end on the satellite at ϵ_p = $ϵ_p_start")

class_up, λ_up, a_up = continuation(s_start, ϵ_p_up, ratio_cut; with_lyap = true)
class_down, λ_down, a_down = continuation(s_start, ϵ_p_down, ratio_cut; stop_on_cat = true, with_lyap = true)

i_lost = findfirst(!=(1.0), class_down)
!isnothing(i_lost) && i_lost > 1 && println("satellite lost going down between ϵ_p = ", ϵ_p_down[i_lost-1], " and ", ϵ_p_down[i_lost],
    " (g2/2π = ", round(g2_of(ϵ_p_down[i_lost])/2π*1e3, digits=1), " MHz), class reached: ", class_down[i_lost])
i_hb = hb_edge(sort(ϵ_p_down), ratio_cut)
i_hb <= length(ϵ_p_down) && println("analytic satellite exists from ϵ_p = ", sort(ϵ_p_down)[i_hb],
    " (g2/2π = ", round(g2_of(sort(ϵ_p_down)[i_hb])/2π*1e3, digits=1), " MHz)")
println("going up, first ϵ_p of each class reached: ",
    Dict(c => ϵ_p_up[findfirst(==(c), class_up)] for c in unique(filter(!isnan, class_up))))

In [ ]:
# bifurcation diagram: Re a at the last n_strobe multiples of T_a, colored by class
p_bif = plot(ylabel=L"\mathrm{Re}\,a(nT_a)", title=L"\varepsilon_d = "*string(ratio_cut)*L"\,g_2"*", continuation from "*
             L"g_2/2\pi = "*string(round(g2_MHz(ϵ_p_start), digits=1))*" MHz")
for (path, cls, a_s, ms) in ((ϵ_p_up, class_up, a_up, :circle), (ϵ_p_down, class_down, a_down, :utriangle))
    for i in 1:length(path)
        isnan(cls[i]) && continue
        scatter!(p_bif, fill(g2_MHz(path[i]), n_strobe), real.(a_s[i, :]), ms=1.5, msw=0, markershape=ms,
                 color=class_colors[Int(cls[i])], label="")
    end
end
for k in 1:5   # legend for the classes
    scatter!(p_bif, [NaN], [NaN], color=class_colors[k], markershape=:square, msw=0, label=class_names[k])
end

p_λ_cont = plot(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\lambda/\kappa_b")
plot!(p_λ_cont, g2_MHz.(ϵ_p_up), λ_up/κ_b, marker=:circle, ms=2, msw=0, label="going up")
plot!(p_λ_cont, g2_MHz.(ϵ_p_down), λ_down/κ_b, marker=:utriangle, ms=2, msw=0, label="going down")
hline!(p_λ_cont, [0], color=:gray, ls=:dash, label="")

for p in (p_bif, p_λ_cont)
    hill_ok && vline!(p, [hill_at(ratio_cut)], color=:black, lw=2, label=(p === p_bif ? "Hill instability" : ""))
    i_hb <= length(ϵ_p_down) && vline!(p, [g2_MHz(sort(ϵ_p_down)[i_hb])], color=:black, ls=:dash, lw=1.5,
                                        label=(p === p_bif ? "analytic satellite edge" : ""))
    plot!(p, xlims=(g2_MHz(minimum(ϵ_p_down)) - 1, g2_MHz(maximum(ϵ_p_up)) + 1))
end
plot!(p_bif, legend=:bottomleft)     # legends inside, so that the two panels keep the same width
plot!(p_λ_cont, legend=:topleft)
p_cont_cut = plot(p_bif, p_λ_cont, layout=grid(2, 1, heights=[0.65, 0.35]), link=:x,
                  size=(2*size_px_car[1], 1.6*size_px_car[2]), left_margin=5mm, bottom_margin=5mm)
savefig(p_cont_cut, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/continuation_cut_satellite_12_10_2026.pdf")
p_cont_cut

### Parameter continuation on the full grid

Same continuation on the grid of the map. For each $\varepsilon_d/g_2$, from the largest down, the satellite at $\varepsilon_p = 0.55$ is continued
from the previous $\varepsilon_d/g_2$, or seeded analytically if that fails. It is then followed up and down in $\varepsilon_p$. Going down, the sweep
stops once the cat is reached, and lower $\varepsilon_p$ stay blank. $\lambda$ is only computed where the attractor is neither period 1 nor period 2.

In [ ]:
# Same grid as the map. For each ϵ_d/g2 (from the largest), the satellite at ϵ_p_start is continued from the previous ϵ_d/g2
# (or seeded analytically if that fails), then followed up and down in ϵ_p.
i_start = argmin(abs.(ϵ_p_list .- ϵ_p_start))
cont_class = fill(NaN, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))   # NaN: not reached (below the point where the cat took over)
cont_λ = fill(NaN, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
seed_kind = fill("", length(ϵ_d_g2_ratio_list))                       # "continued", "analytic" or "none"
s_prev = nothing
for j in ProgressBar(length(ϵ_d_g2_ratio_list):-1:1)
    r = ϵ_d_g2_ratio_list[j]
    s_0 = nothing
    if !isnothing(s_prev)
        s, a_s, mean_abs_b = run_from_state(s_prev, ϵ_p_list[i_start], r)
        classify(a_s, mean_abs_b) == 1.0 && (s_0 = s; seed_kind[j] = "continued")
    end
    if isnothing(s_0)
        s_0 = satellite_state(ϵ_p_list[i_start], r)
        seed_kind[j] = isnothing(s_0) ? "none" : "analytic"
    end
    global s_prev = s_0
    isnothing(s_0) && continue
    cont_class[j, i_start:end], cont_λ[j, i_start:end], _ = continuation(s_0, ϵ_p_list[i_start:end], r)
    cont_class[j, i_start:-1:1], cont_λ[j, i_start:-1:1], _ = continuation(s_0, ϵ_p_list[i_start:-1:1], r; stop_on_cat = true)
end
println("seed at ϵ_p = ", ϵ_p_list[i_start], ": ", Dict(k => count(==(k), seed_kind) for k in unique(seed_kind)))

In [ ]:
g2_list = E_J.*sin.(ϵ_p_list).*φ_a^2 .*φ_b
α_sq = ϵ_d_g2_ratio_list./2
hb_edge_g2 = [g2_MHz(ϵ_p_list[min(hb_edge(ϵ_p_list, r), length(ϵ_p_list))]) for r in ϵ_d_g2_ratio_list]

p_cont = heatmap(g2_list/2π*1e3, α_sq, cont_class, color=cgrad(class_colors, 5, categorical=true), clims=(0.5, 5.5),
                 colorbar=false, title="Continuation from the satellite at "*L"g_2/2\pi = "*string(round(g2_MHz(ϵ_p_list[i_start]), digits=1))*" MHz")
hill_ok && plot!(p_cont, hill_g2, hill_ratio./2, color=:black, lw=2, label="Hill instability")
plot!(p_cont, hb_edge_g2, α_sq, color=:black, ls=:dash, lw=1.5, label="analytic satellite edge")
for k in 1:5   # legend for the classes
    scatter!(p_cont, [NaN], [NaN], color=class_colors[k], markershape=:square, msw=0, label=class_names[k])
end
plot!(p_cont, xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\varepsilon_d/2g_2", legend=:outerright,
      xlims=(minimum(g2_list/2π*1e3), maximum(g2_list/2π*1e3)), ylims=(minimum(α_sq), maximum(α_sq)),
      size=(1.5*size_px_car[1], size_px_car[2]), bottom_margin=5mm, left_margin=5mm)
savefig(p_cont, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/continuation_map_satellite_12_10_2026.pdf")
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/continuation_class_satellite_12_10_2026.json", "w") do io
    JSON.print(io, cont_class)   # 1 satellite, 2 cat, 3 period 2, 4 other non-chaotic, 5 chaotic, NaN not reached
end
p_cont